# 06 — Réponse du modèle (explication sortie + télé)

**Objectif :** expliquer le score pour un séjour test — facteurs locaux + texte métier + aperçu biais/SHAP.

> Aide à la décision uniquement — **pas un diagnostic clinique**.  
> Prérequis : bundles + features curated ; idéalement même `SejourID` que le notebook 05.


## 1. Imports & chemins


In [ ]:
from __future__ import annotations

from pathlib import Path
import sys

import pandas as pd

ROOT = next(
    p for p in [Path.cwd(), *Path.cwd().parents]
    if (p / "patients.csv").is_file() and (p / "Sujet.md").is_file()
)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
if str(ROOT / "notebooks") not in sys.path:
    sys.path.insert(0, str(ROOT / "notebooks"))

from src.data.paths import ProjectPaths
from notebooks._utils.schema import FEATURES_CLES, SCHEMA_ATTENDU

paths = ProjectPaths(root=ROOT)
pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 140)
print("Racine :", ROOT)
print("Curated :", paths.curated)
print("Models  :", paths.models)


## 2. Reprise du séjour et scoring


In [ ]:
import json
import joblib

from src.model.explain_patient import build_explication_risque_30j, facteurs_risque_patient
from src.model.inference import score_single
from src.model.prepare import split_xy

bundle_sortie = joblib.load(paths.models / "score_sortie_bundle.joblib")
bundle_tele = joblib.load(paths.models / "score_tele_bundle.joblib")
metrics_sortie = json.loads((paths.models / "score_sortie_metrics.json").read_text(encoding="utf-8"))
metrics_tele = json.loads((paths.models / "score_tele_metrics.json").read_text(encoding="utf-8"))

features_sortie = pd.read_parquet(paths.curated / "features_score_sortie.parquet")
features_tele = pd.read_parquet(paths.curated / "features_score_tele.parquet")

# Aligner sur le 1er séjour test (même logique que 05) — ou fixer SEJOUR_ID manuellement
SEJOUR_ID = str(features_sortie.loc[features_sortie["split"] == "test", "SejourID"].iloc[0])
print("SejourID :", SEJOUR_ID)

row_sortie = features_sortie.loc[features_sortie["SejourID"] == SEJOUR_ID]
row_tele = features_tele.loc[features_tele["SejourID"] == SEJOUR_ID]
X_so, _, _ = split_xy(row_sortie)
X_te, _, _ = split_xy(row_tele)

score_so = score_single(bundle_sortie, metrics_sortie, X_so)
score_te = score_single(bundle_tele, metrics_tele, X_te)


## 3. Facteurs de risque + texte métier (les 2 scores)


In [ ]:
reponses = []
for label, score, bundle, X_row in [
    ("sortie", score_so, bundle_sortie, X_so),
    ("tele", score_te, bundle_tele, X_te),
]:
    facteurs = facteurs_risque_patient(modele=score["modele"], bundle=bundle, X_row=X_row)
    texte = build_explication_risque_30j(
        proba_pct=int(round(score["proba"] * 100)),
        seuil_pct=int(round(score["seuil"] * 100)),
        alerte=bool(score["alerte"]),
        facteurs=facteurs,
    )
    print("=" * 60)
    print(f"SCORE {label.upper()} — modèle {score['modele']}")
    print(texte)
    print()
    fact_df = pd.DataFrame(facteurs)[["libelle", "valeur", "impact", "shap", "bar_pct"]] if facteurs else pd.DataFrame()
    display(fact_df)
    reponses.append({"score": label, "texte": texte, "n_facteurs": len(facteurs)})

display(pd.DataFrame(reponses))


## 4. Aperçu SHAP global / biais (si fichiers présents)


In [ ]:
aperçus = []
for score_name in ("score_sortie", "score_tele"):
    for suffix in ("shap.csv", "biais.csv"):
        p = paths.models / f"{score_name}_{suffix}"
        aperçus.append({"fichier": p.name, "present": p.is_file(), "chemin": str(p)})

display(pd.DataFrame(aperçus))

for score_name in ("score_sortie", "score_tele"):
    shap_p = paths.models / f"{score_name}_shap.csv"
    biais_p = paths.models / f"{score_name}_biais.csv"
    if shap_p.is_file():
        print(f"\n— {shap_p.name} (tête) —")
        display(pd.read_csv(shap_p).head(8))
    if biais_p.is_file():
        print(f"\n— {biais_p.name} (tête) —")
        display(pd.read_csv(biais_p).head(8))


## 5. Message final

Ces scores et textes sont une **aide à la décision** pour le parcours de sortie / télésurveillance.  
Ils ne remplacent pas l’avis médical, ni une validation clinique. Données synthétiques CISIA.


In [ ]:
print("✓ Parcours ML jury terminé (nettoyage → préparation → entraînement → lancement → réponse).")
print("SejourID expliqué :", SEJOUR_ID)
